# Reuters: TF-IDF, PCA und hierarchisches Clustering

Exploration von Kai S. Kurono im gemeinsamen Projekt mit Jonaid Aydi.

Vorbereitung: `python prepare_data.py`. Dieses Notebook zeigt die drei Mystery-Texte.
Den vollständigen Methodenvergleich auf dem offiziellen Testsplit erzeugt `python run_analysis.py`.

Die gemeinsame Tokenisierung steht in `stylometry/features.py`. PCA dient ausschließlich der Visualisierung. Die Anpassung erfolgt an 147 Referenzartikeln. Der erwartete Stopwort-Hinweis bleibt sichtbar, um die bisherige Vorverarbeitung beizubehalten.

In [ ]:
import os
import re
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist
import matplotlib.pyplot as plt

## 1. DATEN LADEN

In [ ]:
data_dir = "reuter+50+50/C50train"
authors = ["WilliamKazer", "TimFarrand", "PatriciaCommins"]

# Farben pro Author
color_map = {
    authors[0]: '#FF1493',
    authors[1]: '#FFA500',
    authors[2]: '#ADD8E6'
}

texts_dict = {}  # {author: [list of texts]}
mystery_dict = {}  # {author: mystery_text}

for author in authors:
    author_dir = os.path.join(data_dir, author)
    txt_files = sorted([f for f in os.listdir(author_dir) if f.endswith('.txt')])
    
    if len(txt_files) != 50:
        raise ValueError(f'Expected 50 training articles for {author}. Run prepare_data.py first.')
    texts = []
    for txt_file in txt_files:
        with open(os.path.join(author_dir, txt_file), 'r', encoding='utf-8', errors='ignore') as f:
            texts.append(f.read())
    
    # 1 Mystery-Text beiseitelegen (z.B. der erste)
    mystery_dict[author] = texts[0]
    texts_dict[author] = texts[1:]  # Rest für Training
    
    print(f"{author}: {len(texts_dict[author])} Trainingstexte, 1 Mystery-Text")

## 2. TF-IDF BERECHNEN MIT STEMMING UND WÄHRUNGEN

In [ ]:
from stylometry.features import (stemmer, TOKEN_PATTERN, CURRENCY_PATTERN,
                                 custom_tokenizer_with_stemming, build_word_vectorizer)

# Alle Trainingstexte sammeln + Author-Label
all_texts = []
author_labels = []

for author in authors:
    for text in texts_dict[author]:
        all_texts.append(text)
        author_labels.append(author)


def get_stem_to_token_mapping(texts):
    """
    Erstellt eine Zuordnung von Stems zu Originaltokens sodass man die urspünglichen
    Tokens bei der Analyse sehen kann.
    """
    stem_to_token = {}
    
    for text in texts:
        tokens = TOKEN_PATTERN.findall(text.lower())
        
        for token in tokens:
            if re.match(CURRENCY_PATTERN, token):
                stem_to_token[token] = token
            else:
                stem = stemmer.stem(token)
                if stem not in stem_to_token:
                    stem_to_token[stem] = token
    
    return stem_to_token


# TF-IDF Vectorizer mit Stemming
vectorizer = build_word_vectorizer()

tfidf_matrix = vectorizer.fit_transform(all_texts).toarray()
feature_names = np.array(vectorizer.get_feature_names_out())

# Stem-zu-Token-Mapping für Referenz
stem_to_token_map = get_stem_to_token_mapping(all_texts)

# DataFrame für Training
df_train = pd.DataFrame(
    tfidf_matrix,
    columns=feature_names,
    index=[f"{author}_{i}" for author, i in zip(author_labels, range(len(all_texts)))]
)

print(f"\nTF-IDF Matrix: {tfidf_matrix.shape}")
print(f"Features (Wörter + Währungen): {len(feature_names)}")

currency_count = sum(1 for f in feature_names if re.match(CURRENCY_PATTERN, f))
print(f"Währungs-Features: {currency_count}")

## 3. HCA (HIERARCHICAL CLUSTER ANALYSIS) - DENDROGRAMM

In [ ]:
distances = pdist(tfidf_matrix, metric='euclidean')
linkage_matrix = linkage(distances, method='ward')

labels_with_mystery = []
for i in range(len(author_labels)):
    labels_with_mystery.append(f"{author_labels[i]}_{i}")

fig, ax = plt.subplots(figsize=(15, 20))
dendrogram(
    linkage_matrix,
    orientation="right",
    labels=labels_with_mystery,
    leaf_font_size=8,
    ax=ax
)

plt.title('HCA - Hierarchical Cluster Analysis (Dendrogramm)', fontsize=14, fontweight='bold')
plt.ylabel('Dokumente', fontsize=12)
plt.xlabel('Euklidische Distanz', fontsize=12)

plt.tick_params(
    axis='y',
    which='both',
    left=False,
    right=False,
    labelleft=True
)

plt.tight_layout()
plt.savefig('hca_dendrogram.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Dendrogramm gespeichert: hca_dendrogram.png")

## 4. HKA (PRINCIPAL COMPONENT ANALYSIS) MIT BIPLOT

In [ ]:
pca = PCA(n_components=2, svd_solver="full")
pca_matrix = pca.fit_transform(tfidf_matrix)

print(f"\nPCA Explained Variance:")
print(f"  PC1: {pca.explained_variance_ratio_[0]:.2%}")
print(f"  PC2: {pca.explained_variance_ratio_[1]:.2%}")
print(f"  Total: {sum(pca.explained_variance_ratio_):.2%}")

fig, ax = plt.subplots(figsize=(18, 14))

for author in authors:
    mask = np.array(author_labels) == author
    ax.scatter(
        pca_matrix[mask, 0],
        pca_matrix[mask, 1],
        label=f"{author} (Training)",
        s=100,
        alpha=0.7,
        color=color_map[author],
        marker='o'
    )

for author in authors:
    mystery_text = mystery_dict[author]
    mystery_tfidf = vectorizer.transform([mystery_text]).toarray()
    mystery_pca = pca.transform(mystery_tfidf)
    
    ax.scatter(
        mystery_pca[0, 0],
        mystery_pca[0, 1],
        s=300,
        alpha=0.9,
        color=color_map[author],
        marker='*',
        edgecolors='black',
        linewidths=2,
        label=f"{author} (MYSTERY)"
    )

loadings = pca.components_.T
feature_importance = np.sqrt(np.sum(loadings**2, axis=1))
top_features_idx = np.argsort(feature_importance)[-24:]

scale_factor = 3.5

text_objects = []

for idx in top_features_idx:
    ax.arrow(
        0, 0,
        loadings[idx, 0] * scale_factor,
        loadings[idx, 1] * scale_factor,
        head_width=0.08,      
        head_length=0.08,     
        fc='gray',
        ec='gray',
        alpha=0.5,
        linewidth=1.5
    )
    
    text_obj = ax.text(
        loadings[idx, 0] * scale_factor * 1.15,
        loadings[idx, 1] * scale_factor * 1.15,
        feature_names[idx],
        fontsize=9,
        ha='center',
        va='center',
    )
    text_objects.append(text_obj)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})', fontsize=12, fontweight='bold')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})', fontsize=12, fontweight='bold')
ax.set_title('HKA - PCA Biplot mit Feature-Loadings (Mystery: *)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='best')
ax.grid(alpha=0.3)
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)

plt.tight_layout()
plt.savefig('hka_pca_biplot.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ PCA-Biplot gespeichert: hka_pca_biplot.png")

## 5. MYSTERY-TEXTE KLASSIFIZIEREN

In [ ]:
print("\n" + "="*70)
print("MYSTERY-TEXT KLASSIFIZIERUNG (Closest Distance)")
print("="*70)

results = []

for author in authors:
    mystery_text = mystery_dict[author]
    mystery_tfidf = vectorizer.transform([mystery_text]).toarray()
    
    distances_to_mystery = np.linalg.norm(tfidf_matrix - mystery_tfidf, axis=1)
    
    closest_idx = np.argmin(distances_to_mystery)
    closest_author = author_labels[closest_idx]
    closest_distance = distances_to_mystery[closest_idx]
    
    top_3_idx = np.argsort(distances_to_mystery)[:3]
    
    is_correct = "✓ CORRECT" if closest_author == author else "✗ WRONG"
    
    print(f"\n{'='*70}")
    print(f"MYSTERY-TEXT VON: {author.upper()}")
    print(f"{'='*70}")
    print(f"Vorhersage: {closest_author} | Distanz: {closest_distance:.4f} {is_correct}")
    print(f"\nTop 3 nächste Nachbarn:")
    for rank, idx in enumerate(top_3_idx, 1):
        pred_author = author_labels[idx]
        match = "← MATCH" if pred_author == author else ""
        print(f"  {rank}. {pred_author} (Distanz: {distances_to_mystery[idx]:.4f}) {match}")
    
    results.append({
        'True Author': author,
        'Predicted Author': closest_author,
        'Distance': closest_distance,
        'Correct': is_correct
    })

print(f"\n{'='*70}")
print("ZUSAMMENFASSUNG")
print(f"{'='*70}")
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

## 6. FEATURE-TABELLE (Top Features pro Author)

In [ ]:
print("\n" + "="*70)
print("TOP 10 FEATURE-WÖRTER PRO AUTOR")
print("="*70)

for author in authors:
    author_mask = np.array(author_labels) == author
    author_mean_tfidf = tfidf_matrix[author_mask].mean(axis=0)
    
    top_features_idx = np.argsort(author_mean_tfidf)[-10:]
    
    print(f"\n{author}:")
    for rank, idx in enumerate(reversed(top_features_idx), 1):
        feature = feature_names[idx]
        original = stem_to_token_map.get(feature, feature)
        if feature == original:
            print(f"  {rank:2d}. {feature:20s} {author_mean_tfidf[idx]:.4f}")
        else:
            print(f"  {rank:2d}. {feature:20s} (Original: {original:15s}) {author_mean_tfidf[idx]:.4f}")

print("\n" + "="*70)
print("="*70)